In [ ]:
## country order endogenous to the matrix

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list, set_link_color_palette, fcluster
from scipy.spatial.distance import squareform

# ============================================================
# 0. Function: pairwise RMS distance with missing values
# ============================================================
def pairwise_rms_with_missing(X: pd.DataFrame) -> pd.DataFrame:
    """
    Computes the pairwise root mean squared distance between the rows of X,
    using only the columns observed for both rows.
    """
    X_values = X.values.astype(float)
    n = X_values.shape[0]
    dist = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            xi = X_values[i, :]
            xj = X_values[j, :]

            mask = ~np.isnan(xi) & ~np.isnan(xj)

            if mask.sum() == 0:
                d = np.nan
            else:
                diff = xi[mask] - xj[mask]
                d = np.sqrt(np.mean(diff ** 2))

            dist[i, j] = d
            dist[j, i] = d

    np.fill_diagonal(dist, 0.0)
    return pd.DataFrame(dist, index=X.index, columns=X.index)


# ============================================================
# 1. Load data and prepare REAL / LLM
# ============================================================
df = pd.read_excel("country_dataset.xlsx")

df = df[df["Country_match"].notna()]
df = df[df["Country_match"].astype(str).str.strip() != ""].copy()
df["Country_match"] = df["Country_match"].astype(str)
df = df.set_index("Country_match")

real_cols = [c for c in df.columns if c.startswith("REAL")]
llm_cols  = [c for c in df.columns if c.startswith("LLM")]

real_data = df[real_cols].astype(float)
llm_data  = df[llm_cols].astype(float)

# ============================================================
# 2. Predefined initial country order
#    Here: alphabetical order
# ============================================================
initial_order = sorted(real_data.index.tolist())

real_data = real_data.loc[initial_order]
llm_data  = llm_data.loc[initial_order]

# ============================================================
# 3. RMS distance matrices: REAL and LLM
# ============================================================
dist_REAL = pairwise_rms_with_missing(real_data)
dist_LLM  = pairwise_rms_with_missing(llm_data)

# ============================================================
# 4. Difference matrix between distances: REAL - LLM
# ============================================================
diff_matrix = dist_REAL - dist_LLM

# Make sure it follows the desired initial order
diff_matrix = diff_matrix.loc[initial_order, initial_order]

# ============================================================
# 5. Build, for each country, the vector of distance differences
#    with respect to all other countries
# ============================================================
# Each row of diff_matrix is already exactly that vector.
country_diff_vectors = diff_matrix.copy()

# ============================================================
# 6. Compute distances between these difference vectors
# ============================================================
dist_on_diff_vectors = pairwise_rms_with_missing(country_diff_vectors)

# ============================================================
# 7. Hierarchical clustering on the difference vectors
# ============================================================
dist_for_clust = dist_on_diff_vectors.copy()

# linkage does not accept NaN: replace them with the maximum observed distance
max_dist = np.nanmax(dist_for_clust.values)
dist_for_clust = dist_for_clust.fillna(max_dist)

Z_order = linkage(squareform(dist_for_clust.values), method="average")

leaf_order = leaves_list(Z_order)
ordered_labels = dist_on_diff_vectors.index[leaf_order]

# Reorder the final difference matrix according to the dendrogram order
diff_matrix_ord = diff_matrix.loc[ordered_labels, ordered_labels]

# ============================================================
# 8. Plotting parameters
# ============================================================
cmap_diff = LinearSegmentedColormap.from_list(
    "blue_white_red",
    ["blue", "white", "red"]
)

vmax = np.nanmax(np.abs(diff_matrix_ord.values))
vmin = -vmax

plot_data = np.ma.masked_invalid(diff_matrix_ord.values)

# ============================================================
# 9. Cluster parameters
# ============================================================
n_clusters = 4
palette = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'cyan', 'magenta']
set_link_color_palette(palette)

color_thr = Z_order[-n_clusters, 2]

cluster_labels = fcluster(Z_order, t=color_thr, criterion="distance")
for cl in np.unique(cluster_labels):
    members = dist_on_diff_vectors.index[cluster_labels == cl]
    print(f"Cluster {cl}: {list(members)}")

# ============================================================
# 10. Combined figure
# ============================================================
fig = plt.figure(figsize=(18, 12))

gs = fig.add_gridspec(
    1, 3,
    width_ratios=[3.0, 2.3, 10.0],
    wspace=0.01
)

ax_dendro = fig.add_subplot(gs[0, 0])
ax_names  = fig.add_subplot(gs[0, 1])
ax_hm     = fig.add_subplot(gs[0, 2])

dendrogram(
    Z_order,
    orientation="left",
    no_labels=True,
    color_threshold=color_thr,
    ax=ax_dendro
)

ax_dendro.axvline(color_thr, color="black", linestyle="--", linewidth=1)
ax_dendro.set_yticks([])
ax_dendro.set_xlabel("Distance")

ax_names.set_xlim(0, 1)
ax_names.set_ylim(-0.5, len(ordered_labels) - 0.5)
ax_names.axis("off")
for i, lab in enumerate(ordered_labels):
    ax_names.text(0.5, i, str(lab), va="center", ha="center", fontsize=10)

im = ax_hm.imshow(
    plot_data,
    cmap=cmap_diff,
    vmin=vmin,
    vmax=vmax,
    interpolation="nearest",
    aspect="auto",
    origin="lower"
)

ax_hm.set_yticks([])
ax_hm.set_xticks(np.arange(len(ordered_labels)))
ax_hm.set_xticklabels(ordered_labels, rotation=75, ha="right", fontsize=10)

# ============================================================
# 11. Colorbar
# ============================================================
fig.canvas.draw()
pos = ax_hm.get_position()
ncols = diff_matrix_ord.shape[1]
cell_width_fig = pos.width / ncols

gap = cell_width_fig
cb_width = cell_width_fig

cb_left = pos.x1 + gap
cb_bottom = pos.y0
cb_height = pos.height

cax = fig.add_axes([cb_left, cb_bottom, cb_width, cb_height])
fig.colorbar(im, cax=cax, orientation="vertical")

# ============================================================
# 12. Saving
# ============================================================
fig.savefig(
    "dendrogram_plus_DIFF_RMS_matrix_ordered_from_initial_order_vectors.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Done! Saved:")
print(" - dendrogram_plus_DIFF_RMS_matrix_ordered_from_initial_order_vectors.png")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list, set_link_color_palette, fcluster
from scipy.spatial.distance import squareform

# =========================
# 0. Function: pairwise RMS distance with missing values
# =========================
def pairwise_rms_with_missing(X: pd.DataFrame) -> pd.DataFrame:
    X_values = X.values.astype(float)
    n = X_values.shape[0]
    dist = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            xi = X_values[i, :]
            xj = X_values[j, :]

            mask = ~np.isnan(xi) & ~np.isnan(xj)

            if mask.sum() == 0:
                d = np.nan
            else:
                diff = xi[mask] - xj[mask]
                d = np.sqrt(np.mean(diff ** 2))   # root mean squared distance

            dist[i, j] = d
            dist[j, i] = d

    np.fill_diagonal(dist, 0.0)
    return pd.DataFrame(dist, index=X.index, columns=X.index)


# =========================
# 1. Load data and prepare REAL / LLM
# =========================
df = pd.read_excel("country_dataset.xlsx")

df = df[df["Country_match"].notna()]
df = df[df["Country_match"].astype(str).str.strip() != ""].copy()
df["Country_match"] = df["Country_match"].astype(str)
df = df.set_index("Country_match")

real_cols = [c for c in df.columns if c.startswith("REAL")]
llm_cols  = [c for c in df.columns if c.startswith("LLM")]

real_data = df[real_cols].astype(float)
llm_data  = df[llm_cols].astype(float)

# =========================
# 2. Predefined initial country order
#    Here: alphabetical order
# =========================
initial_order = sorted(real_data.index.tolist())

real_data = real_data.loc[initial_order]
llm_data  = llm_data.loc[initial_order]

# =========================
# 3. RMS distance matrices: REAL and LLM
# =========================
dist_REAL = pairwise_rms_with_missing(real_data)
dist_LLM  = pairwise_rms_with_missing(llm_data)

# =========================
# 4. Difference matrix between distances: REAL - LLM
# =========================
diff_matrix = dist_REAL - dist_LLM
diff_matrix = diff_matrix.loc[initial_order, initial_order]

# =========================
# 5. Difference vectors for each country
#    Each row of diff_matrix is the country's vector
# =========================
country_diff_vectors = diff_matrix.copy()

# =========================
# 6. Distance between these difference vectors
# =========================
dist_on_diff_vectors = pairwise_rms_with_missing(country_diff_vectors)

# =========================
# 7. Hierarchical clustering on the difference vectors
# =========================
dist_for_clust = dist_on_diff_vectors.copy()

max_dist = np.nanmax(dist_for_clust.values)
dist_for_clust = dist_for_clust.fillna(max_dist)

Z_order = linkage(squareform(dist_for_clust.values), method="average")

leaf_order = leaves_list(Z_order)
ordered_labels = dist_on_diff_vectors.index[leaf_order]

# Reorder both matrices according to the same ordering
dist_REAL_ord = dist_REAL.loc[ordered_labels, ordered_labels]
dist_LLM_ord  = dist_LLM.loc[ordered_labels, ordered_labels]

# =========================
# 8. Cluster parameters (same mechanism as the previous code)
# =========================
n_clusters = 4
palette = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'cyan', 'magenta']
set_link_color_palette(palette)

color_thr = Z_order[-n_clusters, 2]

cluster_labels = fcluster(Z_order, t=color_thr, criterion="distance")
for cl in np.unique(cluster_labels):
    members = dist_on_diff_vectors.index[cluster_labels == cl]
    print(f"Cluster {cl}: {list(members)}")

# =========================
# 9. Heatmap colormap
# =========================
cmap = LinearSegmentedColormap.from_list(
    "white_to_darkred",
    ["white", "darkred"]
)

# =========================
# 10. Function to save the combined figure:
#     colored dendrogram | labels | heatmap | colorbar
# =========================
def plot_distance_matrix_with_dendrogram(
    dist_matrix_ord: pd.DataFrame,
    Z_order,
    ordered_labels,
    color_thr: float,
    output_name: str
):
    plot_data = np.ma.masked_invalid(dist_matrix_ord.values)

    vmin = 0
    vmax = np.nanmax(dist_matrix_ord.values)

    fig = plt.figure(figsize=(18, 12))

    gs = fig.add_gridspec(
        1, 3,
        width_ratios=[3.0, 2.3, 10.0],
        wspace=0.01
    )

    ax_dendro = fig.add_subplot(gs[0, 0])
    ax_names  = fig.add_subplot(gs[0, 1])
    ax_hm     = fig.add_subplot(gs[0, 2])

    # Dendrogram using the same cluster coloring mechanism
    dendrogram(
        Z_order,
        orientation="left",
        no_labels=True,
        color_threshold=color_thr,
        ax=ax_dendro
    )

    ax_dendro.axvline(color_thr, color="black", linestyle="--", linewidth=1)
    ax_dendro.set_yticks([])
    ax_dendro.set_xlabel("Distance")

    # Labels
    ax_names.set_xlim(0, 1)
    ax_names.set_ylim(-0.5, len(ordered_labels) - 0.5)
    ax_names.axis("off")
    for i, lab in enumerate(ordered_labels):
        ax_names.text(0.5, i, str(lab), va="center", ha="center", fontsize=10)

    # Heatmap
    im = ax_hm.imshow(
        plot_data,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        aspect="auto",
        origin="lower"
    )

    ax_hm.set_yticks([])
    ax_hm.set_xticks(np.arange(len(ordered_labels)))
    ax_hm.set_xticklabels(ordered_labels, rotation=75, ha="right", fontsize=10)

    # Colorbar as wide as a single cell
    fig.canvas.draw()
    pos = ax_hm.get_position()
    ncols = dist_matrix_ord.shape[1]
    cell_width_fig = pos.width / ncols

    gap = cell_width_fig
    cb_width = cell_width_fig

    cb_left = pos.x1 + gap
    cb_bottom = pos.y0
    cb_height = pos.height

    cax = fig.add_axes([cb_left, cb_bottom, cb_width, cb_height])
    fig.colorbar(im, cax=cax, orientation="vertical")

    fig.savefig(output_name, dpi=300, bbox_inches="tight")
    plt.show()


# =========================
# 11. Save REAL matrix
# =========================
plot_distance_matrix_with_dendrogram(
    dist_matrix_ord=dist_REAL_ord,
    Z_order=Z_order,
    ordered_labels=ordered_labels,
    color_thr=color_thr,
    output_name="dendrogram_distance_REAL_RMS_ordered_by_diff_vectors.png"
)

# =========================
# 12. Save LLM matrix
# =========================
plot_distance_matrix_with_dendrogram(
    dist_matrix_ord=dist_LLM_ord,
    Z_order=Z_order,
    ordered_labels=ordered_labels,
    color_thr=color_thr,
    output_name="dendrogram_distance_LLM_RMS_ordered_by_diff_vectors.png"
)

print("Done! Saved:")
print(" - dendrogram_distance_REAL_RMS_ordered_by_diff_vectors.png")
print(" - dendrogram_distance_LLM_RMS_ordered_by_diff_vectors.png")